In [ ]:
!pip install langchain-classic transformers==4.52.4 torch

In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM
import torch

model_name = "Qwen/Qwen2.5-1.5B-Instruct"
llm_tokenizer = AutoTokenizer.from_pretrained(model_name)
llm_model = AutoModelForCausalLM.from_pretrained(
    model_name,
    torch_dtype="auto",
    device_map="auto"
)

def generate_text(prompt: str, max_new_tokens: int = 512) -> str:
    """
    Generates a single string response from the Qwen model given a prompt.
    Handles chat templating, device mapping, and formatting.
    """
    messages = [{"role": "user", "content": prompt}]
    text = llm_tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )

    model_inputs = llm_tokenizer([text], return_tensors="pt").to(llm_model.device)

    generated_ids = llm_model.generate(
        **model_inputs,
        max_new_tokens=max_new_tokens,
        do_sample=False
    )

    # Slice out the prompt tokens so we only decode the newly generated tokens
    new_tokens = [
        output_ids[len(input_ids):]
        for input_ids, output_ids in zip(model_inputs.input_ids, generated_ids)
    ]

    decoded_outputs = llm_tokenizer.batch_decode(new_tokens, skip_special_tokens=True)
    return decoded_outputs[0].strip()


In [ ]:
from langchain_community.document_loaders import PyPDFLoader

# NOTE: update this path if running outside Kaggle
path = "/kaggle/input/datasets/mazenbassem/jordan-cv/sample_cv.pdf"
loader = PyPDFLoader(path)
document = loader.load()

# Join all page contents into a single clean string (avoids dumping
# Document(metadata=..., page_content=...) repr syntax into the prompt)
resume_text = "\n".join(page.page_content for page in document)
print(resume_text[:500])  # quick sanity check, not the full dump


In [ ]:
from langchain_classic.output_parsers.structured import StructuredOutputParser, ResponseSchema
from langchain_classic.prompts import PromptTemplate

response_schemas = [
    ResponseSchema(name="full_name", description="The candidate's full name (first and last)."),
    ResponseSchema(name="email", description="The email address extracted from the resume."),
    ResponseSchema(name="education", description="List of education objects. Each object should have 'degree', 'institution', and 'year' keys. If none, return empty list."),
    ResponseSchema(name="skills", description="A flat list of string values representing candidate's skills."),
    ResponseSchema(name="experience", description="List of experience objects. Each object should have 'role', 'company', and 'years' keys. If none, return empty list."),
]

output_parser = StructuredOutputParser.from_response_schemas(response_schemas)
format_instructions = output_parser.get_format_instructions()
print(format_instructions)


In [ ]:
# Explicit system instructions combined with strict formatting boundaries
ats_template = """You are an accurate Applicant Tracking System (ATS) parsing assistant.
Your task is to extract information from the resume text below and format it strictly as a JSON object matching the provided schema.

Strict Rules:
1. Extract data ONLY from the provided candidate resume text. Do not assume, extrapolate, or invent any facts.
2. If a field is not present or mentioned in the resume text, represent it as an empty array [] or null.
3. Start your response immediately with the opening curly brace "{{".
4. DO NOT wrap your output in markdown formatting backticks (do not write ```json ... ```). Write pure, raw JSON only.
5. Output "Experience" as bullet points, make each one concise, objective sentence while maintaining keywords.

Candidate Resume Text:
{user_input}

Formatting Rules:
{format_instructions}

JSON Output:"""

print(ats_template)


In [ ]:
ats_prompt = PromptTemplate(
    template=ats_template,
    input_variables=["user_input"],
    partial_variables={"format_instructions": format_instructions},
).format(user_input=resume_text)

print(type(ats_prompt))

In [ ]:
response = generate_text(ats_prompt, max_new_tokens=1000)
print(response)

In [ ]:
import json

def strip_code_fences(text: str) -> str:
    """Guards against the model wrapping JSON in ```json ... ``` despite instructions not to."""
    text = text.strip()
    if text.startswith("```"):
        lines = text.splitlines()
        if lines and lines[0].startswith("```"):
            lines = lines[1:]
        text = "\n".join(lines).strip()
    if text.endswith("```"):
        lines = text.splitlines()
        if lines and lines[-1].strip() == "```":
            lines = lines[:-1]
        text = "\n".join(lines).strip()
    return text

clean_response = strip_code_fences(response)

try:
    output_data = output_parser.parse(clean_response)
    print("Parsed successfully:")
    print(json.dumps(output_data, indent=4))
except Exception as e:
    print("Parsing failed. Raw cleaned output:\n", clean_response)
    print("\nParser error:", str(e))

In [ ]:
output_path = "ats_output.json"
with open(output_path, "w") as f:
    json.dump(output_data, f, indent=4)

print(f"Saved parsed result to {output_path}")
